## import library

In [4]:
# for data processing
import numpy as np 
import pandas as pd 

#filter warnings
import warnings
warnings.filterwarnings('ignore')
#set max columns display in pandas 
pd.set_option("display.max_columns", 40)
#to preprocessing data
from sklearn.preprocessing import PowerTransformer,OneHotEncoder,LabelEncoder,MinMaxScaler,OrdinalEncoder  #OneHotEncoder>Multicategory,LabelEncoder>binary,map():know variable
#to balance class imbalance 
from imblearn.over_sampling import SMOTE

#for dividing data
from sklearn.model_selection import train_test_split,GridSearchCV
                         #To use the SVG function from IPython.display, you need to have IPython installed, which usually comes pre-installed with Jupyter Notebook.
from sklearn.ensemble import RandomForestClassifier  #bagging ensemble method 

#for evaluation of model
from sklearn.metrics import accuracy_score,precision_score,recall_score,roc_auc_score,confusion_matrix
#model explainability models for black box
import shap
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from imblearn.pipeline import Pipeline

#import function and class
import sys
import os

sys.path.append(os.path.abspath(".."))
from src.featureengineering import FeatureEngineering
from src.preprocessor import preprocessor

## Import Datasets

In [7]:
# final model creation for ml pipelines
df = pd.read_csv("../datasets/WA_Fn-UseC_-Telco-Customer-Churn.csv")
#divide data into x and y
X = df.drop('Churn', axis=1)
y = df['Churn']
y=y.map({'No':0,'Yes':1})
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=5295,
    stratify=y
)

In [8]:
X.head(1)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85


In [7]:

#Step 1 — Create Custom Transformer (Feature Engineering)

class FeatureEngineering(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):

        X = X.copy()
        
        X = X.drop(columns=['customerID'])
        # convert TotalCharges
        X['TotalCharges'] = pd.to_numeric(X['TotalCharges'], errors='coerce')
        X['TotalCharges'] = X['TotalCharges'].fillna(X['MonthlyCharges'])

        # MonthlyCharges category
        def cal_MonthlyCharges(val):

            if val <= 30:
                return 0
            elif val <= 60:
                return 1
            elif val <= 90:
                return 2
            else:
                return 3

         # binary mapping
        binary_map = {'Yes':1,'No':0}

        for col in ['Partner','Dependents']:
            X[col] = X[col].map(binary_map)

        X['gender'] = X['gender'].map({'Male':1,'Female':0})

        # 3-category mapping
        X['MultipleLines'] = X['MultipleLines'].map({'Yes':1,'No':0,'No phone service':2})
        X['TechSupport'] = X['TechSupport'].map({'Yes':1,'No':0,'No internet service':2})
        X['OnlineSecurity'] = X['OnlineSecurity'].map({'Yes':1,'No':0,'No internet service':2})
        X['OnlineBackup'] = X['OnlineBackup'].map({'Yes':1,'No':0,'No internet service':2})
        X['DeviceProtection'] = X['DeviceProtection'].map({'Yes':1,'No':0,'No internet service':2})
        X['StreamingTV'] = X['StreamingTV'].map({'Yes':1,'No':0,'No internet service':2})
        X['StreamingMovies'] = X['StreamingMovies'].map({'Yes':1,'No':0,'No internet service':2})


        X['MonthlyCharges_category'] = X['MonthlyCharges'].apply(cal_MonthlyCharges)

        return X


#step2-— Column Preprocessing

num_cols = ['MonthlyCharges','tenure']
power_cols = ['TotalCharges']
binary_cols = ['PhoneService','PaperlessBilling']
cat_cols = ['InternetService','Contract','PaymentMethod']

preprocessor = ColumnTransformer(

    transformers=[

        ('power', PowerTransformer(), power_cols),
        ('scale', MinMaxScaler(), num_cols),
        ('binary', OrdinalEncoder(), binary_cols),
        ('onehot', OneHotEncoder(handle_unknown='ignore'), cat_cols)

    ],

    remainder='passthrough'

)


#Pipeline:
pipeline = Pipeline(steps=[

    ('feature_engineering', FeatureEngineering()),

    ('preprocessing', preprocessor),

    ('smote', SMOTE(sampling_strategy={1:3300}, random_state=5)),

    ('model', RandomForestClassifier(
        n_estimators=80,
        max_depth=7,
        random_state=5295
    ))

])



In [9]:
pipeline.fit(X_train, y_train)




NameError: name 'pipeline' is not defined

In [ ]:
#save pipeline

import joblib
import os 

import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(pipeline, "../models/churn_pipeline.pkl")
#load pipeline models
pipeline = joblib.load("../models/churn_pipeline.pkl")

In [21]:
#prediction of model on train data and test data
y_train_pr=pipeline.predict(X_train)
y_test_pr=pipeline.predict(X_test)

#on train data
print('Accuracy of model on  train data :',accuracy_score(y_train,y_train_pr))    #accuracy=(tp+tn)/(tp+tn+fp+fn)
print('Precison of model on train data :',precision_score(y_train,y_train_pr))     #precision=tp/(tp+fp)
print('Recall of model on train data :',recall_score(y_train,y_train_pr))          #recall=tp/(tp+fn)

#on test data
print('Accuracy of model on  test data :',accuracy_score(y_test,y_test_pr))
print('Precison of model on test data :',precision_score(y_test,y_test_pr)) 
print('Recall of model on test data :',recall_score(y_test,y_test_pr)) 

print(roc_auc_score(y_train,y_train_pr))
print(roc_auc_score(y_test,y_test_pr))


Accuracy of model on  train data : 0.8047568335108272
Precison of model on train data : 0.6040021063717746
Recall of model on train data : 0.7672240802675585
Accuracy of model on  test data : 0.7579843860894251
Precison of model on test data : 0.5325443786982249
Recall of model on test data : 0.7219251336898396
0.7927688412934798
0.7464698132217313


In [22]:
pipeline.named_steps['preprocessing'].get_feature_names_out()

array(['power__TotalCharges', 'scale__MonthlyCharges', 'scale__tenure',
       'binary__PhoneService', 'binary__PaperlessBilling',
       'onehot__InternetService_DSL',
       'onehot__InternetService_Fiber optic',
       'onehot__InternetService_No', 'onehot__Contract_Month-to-month',
       'onehot__Contract_One year', 'onehot__Contract_Two year',
       'onehot__PaymentMethod_Bank transfer (automatic)',
       'onehot__PaymentMethod_Credit card (automatic)',
       'onehot__PaymentMethod_Electronic check',
       'onehot__PaymentMethod_Mailed check', 'remainder__gender',
       'remainder__SeniorCitizen', 'remainder__Partner',
       'remainder__Dependents', 'remainder__MultipleLines',
       'remainder__OnlineSecurity', 'remainder__OnlineBackup',
       'remainder__DeviceProtection', 'remainder__TechSupport',
       'remainder__StreamingTV', 'remainder__StreamingMovies',
       'remainder__MonthlyCharges_category'], dtype=object)

In [23]:
pipeline.named_steps['model'].n_features_in_

27